In [ ]:
base_uri = "https://d37ci6vzurychx.cloudfront.net/trip-data"

In [ ]:
file_uri = f"{base_uri}/green_tripdata_2025-04.parquet"

In [ ]:
!curl -O $file_uri

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

In [ ]:
df_full = pd.read_parquet("data/green_tripdata_2025-04.parquet")
df_full

In [ ]:
df_full.info()

In [ ]:
# KPIs: utilization %, avg fare/km, avg idle time, outlier count

In [ ]:
df = df_full[["VendorID", "lpep_pickup_datetime", "lpep_dropoff_datetime"]]
df

In [ ]:
df = df.rename(
    columns={
        "VendorID": "fleet_id",
        "lpep_pickup_datetime": "dt_pickup",
        "lpep_dropoff_datetime": "dt_dropoff",
    }
)
df.info()

In [ ]:
df.describe()

In [ ]:
num_drivers = int(len(df) / 300)
fleets_ids = np.random.randint(1e6, 1e7 - 1, num_drivers)
num_drivers, np.unique(fleets_ids).shape

In [ ]:
from sklearn.utils import resample

In [ ]:
drivers_ids = resample(fleets_ids, n_samples=len(df))

In [ ]:
df["driver_id"] = drivers_ids

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
from datetime import datetime

In [ ]:
df["day"] = df.dt_pickup.dt.floor("D")

In [ ]:
df["duration"] = (df.dt_dropoff - df.dt_pickup).apply(lambda td: td.total_seconds())

In [ ]:
df

In [ ]:
df = df.drop(columns=["dt_pickup", "dt_dropoff"])
df

In [ ]:
df["fleet_id"].unique()

In [ ]:
df_driver_stats = (
    df.groupby(["fleet_id", "driver_id", "day"])
    .agg(
        utilization=("duration", "sum"),
    )
    .reset_index()
)
df_driver_stats

In [ ]:
shift_duration = 86400
df_driver_stats["utilization_rate"] = df_driver_stats["utilization"] / shift_duration
df_driver_stats

In [ ]:
df_fleet_stats = (
    df_driver_stats.groupby(["fleet_id", "day"])
    .agg(
        utilization_rate=("utilization_rate", "mean"),
    )
    .reset_index()
)
df_fleet_stats

In [ ]:
df_fleet_stats["fleet_id"] = df_fleet_stats["fleet_id"].apply(str)
df_fleet_stats.info()

In [ ]:
px.bar(df_fleet_stats, x="day", y="utilization_rate", color="fleet_id")

In [ ]:
df_stats_1 = df_fleet_stats[df_fleet_stats["fleet_id"] == "2"]

In [ ]:
df_stats_1["zscore"] = (
    df_stats_1["utilization_rate"] - df_stats_1["utilization_rate"].mean()
) / df_stats_1["utilization_rate"].std()
df_stats_1["is_outlier"] = df_stats_1["zscore"].apply(lambda z: z > 3.0)
px.scatter(df_stats_1, x="day", y="utilization_rate", color="is_outlier")